**Nafiu Ikeoluwa HAMMED - Homework - More Data Preparation and FINAL VISUALIZATION USING DASH**

**Key Features and metrics in this Dashboard with Hover - using Dash Plotly**
1. *Fixed Cards:*
- Displays the total count of distinct teams and players.
2. *Visualization Dropdown:*

**(A)** Users can select different visualizations with hover:
- **Cap Value and Cash Spent Distribution by Position** – A grouped Bar chart showing how cap values and cash spent vary across different player positions
- **Team Cap Value and Cash Spent Comparison** – A comparative Bar chart displaying how teams allocate cap values and cash
- **Player Touchdowns Vs. Cap Value** – A Scatter plot examining the relationship between a team's cap value and their number of touchdowns
- **Player Efficiency Across Teams** – A Scatter plot comparing player efficiency (cash spent per touchdown) across different teams
- **Top 10 Teams Based on Cap Value and Cash Spent** - Bar charts showing the top 10 teams ranked by cap value and cash spent
- **Bottom 10 Teams Based on Cap Value and Cash Spent** - Bar charts showing the bottom 10 teams ranked by cap value and cash spent
- **Top 10 Players Based on Cap Value and Cash Spent** - Bar charts showing the top 10 players ranked by cap value and cash spent
- **Bottom 10 Players Based on Cap Value and Cash Spent** - Bar charts showing the bottom 10 players ranked by cap value and cash spent

**(B)** Tables
- **Top 10 Players Table** - Displays the top 10 Players based on touchdowns along with their position, their team, cap value and cash spent.

- **Top 10 Teams Table** - Displays the top 10 Teams based on touchdowns along with their cap value and cash spent.


In [1]:
!pip install dash

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 19.9 MB/s eta 0:00:00


In [2]:
!pip install dash jupyter-dash

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 13.6 MB/s eta 0:00:00


### Dataset location
For the GitHub portfolio version, the dataset is stored inside the repository at `data/prepared_football_statistics.xlsx`, so Google Drive mounting is not required.


In [6]:
import plotly.express as px
import pandas as pd
import numpy as np
from dash import Dash, dcc, html, Input, Output, dash_table


# -------------------------------------------------------
# Load prepared dataset
# -------------------------------------------------------
from pathlib import Path

file_path = Path('data') / 'prepared_football_statistics.xlsx'

if not file_path.exists():
    raise FileNotFoundError(
        f"Dataset not found at {file_path}. Run the notebook from the repository root."
    )

data = pd.read_excel(file_path)


# -------------------------------------------------------
# Make sure required numeric columns are numeric
# -------------------------------------------------------
numeric_columns = ['Cap Value', 'Cash Spent', 'Touchdowns']

for column in numeric_columns:
    data[column] = pd.to_numeric(data[column], errors='coerce')

# Replace missing numeric values with 0
data[numeric_columns] = data[numeric_columns].fillna(0)


# -------------------------------------------------------
# Aggregate data for visualizations
# -------------------------------------------------------
distinct_teams = data['Team'].nunique()
distinct_players = data['PlayerName'].nunique()

position_summary = (
    data.groupby('Position', as_index=False)
    .agg({
        'Cap Value': 'sum',
        'Cash Spent': 'sum'
    })
)

team_summary = (
    data.groupby('Team', as_index=False)
    .agg({
        'Cap Value': 'sum',
        'Cash Spent': 'sum',
        'Touchdowns': 'sum'
    })
)


# -------------------------------------------------------
# Calculate player efficiency
# Prevent division by zero
# -------------------------------------------------------
efficiency = data.copy()

efficiency['Efficiency'] = np.where(
    efficiency['Cap Value'] > 0,
    efficiency['Touchdowns'] / efficiency['Cap Value'],
    np.nan
)


# -------------------------------------------------------
# Top/Bottom 10 Teams
# -------------------------------------------------------
top_10_teams = (
    team_summary
    .sort_values(by='Cap Value', ascending=False)
    .head(10)
)

bottom_10_teams = (
    team_summary
    .sort_values(by='Cap Value', ascending=True)
    .head(10)
)


# -------------------------------------------------------
# Top/Bottom 10 Players
# -------------------------------------------------------
top_10_players_cap = (
    data
    .sort_values(by='Cap Value', ascending=False)
    .head(10)
)

bottom_10_players_cap = (
    data
    .sort_values(by='Cap Value', ascending=True)
    .head(10)
)


# -------------------------------------------------------
# Top 10 Teams by Touchdowns
# -------------------------------------------------------
top_10_teams_touchdown = (
    team_summary
    .sort_values(by='Touchdowns', ascending=False)
    .head(10)
)


# -------------------------------------------------------
# Top 10 Players by Touchdowns
# -------------------------------------------------------
top_10_touchdowns = (
    data
    .sort_values(by='Touchdowns', ascending=False)
    .head(10)
)


# -------------------------------------------------------
# Initialize Dash app
# -------------------------------------------------------
app = Dash(__name__)


# -------------------------------------------------------
# Define Dashboard Layout
# -------------------------------------------------------
app.layout = html.Div([

    html.H1(
        "Interactive Football Statistics Dashboard "
        "By Nafiu Ikeoluwa HAMMED - VISUALIZATION",
        style={'textAlign': 'center'}
    ),

    # ---------------------------------------------------
    # Cards for summary statistics
    # ---------------------------------------------------
    html.Div([

        html.Div([
            html.H3("Overall Distinct Teams"),
            html.P(f"{distinct_teams}")
        ], style={
            'width': '45%',
            'display': 'inline-block',
            'textAlign': 'center'
        }),

        html.Div([
            html.H3("Overall Distinct Players"),
            html.P(f"{distinct_players}")
        ], style={
            'width': '45%',
            'display': 'inline-block',
            'textAlign': 'center'
        }),

    ], style={'marginBottom': '20px'}),


    # ---------------------------------------------------
    # Dropdown for visualization selection
    # ---------------------------------------------------
    dcc.Dropdown(
        id='visualization-choice',

        options=[
            {
                'label': 'Cap Value and Cash Spent Distribution by Position',
                'value': 'position'
            },
            {
                'label': 'Team Cap Value and Cash Spent Comparison',
                'value': 'team'
            },
            {
                'label': 'Touchdowns vs. Cap Value',
                'value': 'touchdowns_vs_cap'
            },
            {
                'label': 'Player Efficiency Across Teams',
                'value': 'efficiency'
            },
            {
                'label': 'Top 10 Teams by Cap Value',
                'value': 'top_10_teams'
            },
            {
                'label': 'Bottom 10 Teams by Cap Value',
                'value': 'bottom_10_teams'
            },
            {
                'label': 'Top 10 Players by Cap Value',
                'value': 'top_10_players'
            },
            {
                'label': 'Bottom 10 Players by Cap Value',
                'value': 'bottom_10_players'
            },
            {
                'label': 'Top 10 Teams by Touchdowns (Table)',
                'value': 'top_10_teamtouchdowns_table'
            },
            {
                'label': 'Top 10 Players by Touchdowns (Table)',
                'value': 'top_10_touchdowns_table'
            }
        ],

        value='position',

        clearable=False,

        style={
            'width': '60%',
            'marginBottom': '20px'
        }
    ),


    # ---------------------------------------------------
    # Graph area
    # ---------------------------------------------------
    dcc.Graph(
        id='visualization-graph',
        style={'display': 'block'}
    ),


    # ---------------------------------------------------
    # Table area
    # ---------------------------------------------------
    dash_table.DataTable(
        id='visualization-table',

        data=[],
        columns=[],

        style_table={
            'marginTop': '10px',
            'display': 'none',
            'overflowX': 'auto'
        },

        style_cell={
            'textAlign': 'left',
            'padding': '8px'
        },

        style_header={
            'fontWeight': 'bold'
        },

        page_size=10
    )

])


# -------------------------------------------------------
# Callback to update graph or table
# -------------------------------------------------------
@app.callback(
    [
        Output('visualization-graph', 'figure'),
        Output('visualization-graph', 'style'),
        Output('visualization-table', 'data'),
        Output('visualization-table', 'columns'),
        Output('visualization-table', 'style_table')
    ],

    Input('visualization-choice', 'value')
)

def update_visualization(selected_value):

    # ---------------------------------------------------
    # Position chart
    # ---------------------------------------------------
    if selected_value == 'position':

        fig = px.bar(
            position_summary,
            x='Position',
            y=['Cap Value', 'Cash Spent'],
            barmode='group',
            title='Cap Value ($) and Cash Spent ($) Distribution by Position'
        )

        return (
            fig,
            {'display': 'block'},
            [],
            [],
            {'display': 'none'}
        )


    # ---------------------------------------------------
    # Team chart
    # ---------------------------------------------------
    elif selected_value == 'team':

        fig = px.bar(
            team_summary,
            x='Team',
            y=['Cap Value', 'Cash Spent'],
            barmode='group',
            title='Team Cap Value ($) and Cash Spent ($) Comparison'
        )

        return (
            fig,
            {'display': 'block'},
            [],
            [],
            {'display': 'none'}
        )


    # ---------------------------------------------------
    # Touchdowns vs Cap Value
    # ---------------------------------------------------
    elif selected_value == 'touchdowns_vs_cap':

        fig = px.scatter(
            data,
            x='Cap Value',
            y='Touchdowns',
            title='Player Touchdowns vs. Cap Value ($)',
            hover_data=['PlayerName', 'Team', 'Position']
        )

        return (
            fig,
            {'display': 'block'},
            [],
            [],
            {'display': 'none'}
        )


    # ---------------------------------------------------
    # Player Efficiency
    # ---------------------------------------------------
    elif selected_value == 'efficiency':

        fig = px.scatter(
            efficiency,
            x='Team',
            y='Efficiency',
            title='Player Efficiency Across Teams',
            hover_data=[
                'PlayerName',
                'Position',
                'Touchdowns',
                'Cap Value'
            ]
        )

        return (
            fig,
            {'display': 'block'},
            [],
            [],
            {'display': 'none'}
        )


    # ---------------------------------------------------
    # Top 10 Teams
    # ---------------------------------------------------
    elif selected_value == 'top_10_teams':

        fig = px.bar(
            top_10_teams,
            x='Team',
            y='Cap Value',
            title='Top 10 Teams by Cap Value'
        )

        return (
            fig,
            {'display': 'block'},
            [],
            [],
            {'display': 'none'}
        )


    # ---------------------------------------------------
    # Bottom 10 Teams
    # ---------------------------------------------------
    elif selected_value == 'bottom_10_teams':

        fig = px.bar(
            bottom_10_teams,
            x='Team',
            y='Cap Value',
            title='Bottom 10 Teams by Cap Value'
        )

        return (
            fig,
            {'display': 'block'},
            [],
            [],
            {'display': 'none'}
        )


    # ---------------------------------------------------
    # Top 10 Players
    # ---------------------------------------------------
    elif selected_value == 'top_10_players':

        fig = px.bar(
            top_10_players_cap,
            x='PlayerName',
            y='Cap Value',
            title='Top 10 Players by Cap Value',
            hover_data=['Position', 'Team', 'Cash Spent']
        )

        return (
            fig,
            {'display': 'block'},
            [],
            [],
            {'display': 'none'}
        )


    # ---------------------------------------------------
    # Bottom 10 Players
    # ---------------------------------------------------
    elif selected_value == 'bottom_10_players':

        fig = px.bar(
            bottom_10_players_cap,
            x='PlayerName',
            y='Cap Value',
            title='Bottom 10 Players by Cap Value',
            hover_data=['Position', 'Team', 'Cash Spent']
        )

        return (
            fig,
            {'display': 'block'},
            [],
            [],
            {'display': 'none'}
        )


    # ---------------------------------------------------
    # Top 10 Teams by Touchdowns TABLE
    # ---------------------------------------------------
    elif selected_value == 'top_10_teamtouchdowns_table':

        formatted_data = top_10_teams_touchdown.copy()

        formatted_data['Cap Value'] = formatted_data['Cap Value'].apply(
            lambda x: f"${x:,.2f}"
        )

        formatted_data['Cash Spent'] = formatted_data['Cash Spent'].apply(
            lambda x: f"${x:,.2f}"
        )

        table_columns = [
            {'name': 'Team', 'id': 'Team'},
            {'name': 'Cap Value', 'id': 'Cap Value'},
            {'name': 'Cash Spent', 'id': 'Cash Spent'},
            {'name': 'Touchdowns', 'id': 'Touchdowns'}
        ]

        return (
            {},
            {'display': 'none'},
            formatted_data.to_dict('records'),
            table_columns,
            {
                'display': 'block',
                'marginTop': '10px',
                'overflowX': 'auto'
            }
        )


    # ---------------------------------------------------
    # Top 10 Players by Touchdowns TABLE
    # ---------------------------------------------------
    elif selected_value == 'top_10_touchdowns_table':

        formatted_data = top_10_touchdowns.copy()

        formatted_data['Cap Value'] = formatted_data['Cap Value'].apply(
            lambda x: f"${x:,.2f}"
        )

        formatted_data['Cash Spent'] = formatted_data['Cash Spent'].apply(
            lambda x: f"${x:,.2f}"
        )

        table_columns = [
            {'name': 'Player Name', 'id': 'PlayerName'},
            {'name': 'Position', 'id': 'Position'},
            {'name': 'Team', 'id': 'Team'},
            {'name': 'Cap Value', 'id': 'Cap Value'},
            {'name': 'Cash Spent', 'id': 'Cash Spent'},
            {'name': 'Touchdowns', 'id': 'Touchdowns'}
        ]

        return (
            {},
            {'display': 'none'},
            formatted_data.to_dict('records'),
            table_columns,
            {
                'display': 'block',
                'marginTop': '10px',
                'overflowX': 'auto'
            }
        )


    # Default return
    return (
        {},
        {'display': 'none'},
        [],
        [],
        {'display': 'none'}
    )



if __name__ == '__main__':
    app.run(jupyter_mode='inline', debug=True)

<IPython.core.display.Javascript object>